# YOLOv8: распознавание документов (IDD)

В этом ноутбуке:
1. готовится датасет из COCO-разметки Roboflow в формат YOLO;
2. обучается модель **YOLOv8**;
3. модель валидируется на `valid` и тестируется на `test`;
4. строятся метрики, матрица ошибок и визуализация предсказаний.

**Датасет:** `idd.v1i.coco` — 3549 изображений 640×640.

По умолчанию включён пресет **`passport_fields`** — 25 классов полей паспорта
(как на греческом паспорте). Для остальных 52 классов установите `TRAIN_PRESET = "all"`.

> Устройство выбирается автоматически: `cuda:0`, если доступен GPU, иначе `cpu`.
> На CPU обучение заметно медленнее — уменьшите `EPOCHS` или размер выборки.

In [ ]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from ultralytics import YOLO

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)

# --- пути -------------------------------------------------------------------
ROOT = Path.cwd()
if not (ROOT / "idd.v1i.coco").exists() and (ROOT.parent / "idd.v1i.coco").exists():
    ROOT = ROOT.parent

SOURCE_DATASET = ROOT / "idd.v1i.coco"
PREPARE_SCRIPT = ROOT / "scripts" / "prepare_dataset.py"

# --- выбор классов ----------------------------------------------------------
# "passport_fields" — только поля паспорта (25 классов, 1200 изображений, split 80/10/10)
# "all"            — все 52 класса и исходные split'ы
TRAIN_PRESET = "passport_fields"
PRESET_OUTPUT = {
    "all": ROOT / "datasets" / "idd",
    "passport_fields": ROOT / "datasets" / "idd_passport",
}
FORCE_REPREPARE = False
DATASET_DIR = PRESET_OUTPUT[TRAIN_PRESET]
DATA_YAML = DATASET_DIR / "data.yaml"

# --- гиперпараметры ---------------------------------------------------------
MODEL_NAME = "yolov8n.pt"   # yolov8n / yolov8s / yolov8m / yolov8l / yolov8x
EPOCHS = 50                 # на CPU попробуйте 10-20
IMGSZ = 640
# WSL2 ограничивает pinned host memory: validation использует workers*2 процессов.
# Поэтому в WSL запускаем без multiprocessing, иначе возможен CUDA OOM в pin_memory.
IS_WSL = "microsoft" in Path("/proc/version").read_text(encoding="utf-8").lower()
BATCH = 8 if IS_WSL else 16
WORKERS = 0 if (os.name == "nt" or IS_WSL) else 8
SEED = 42
PROJECT = ROOT / "runs" / "detect"
RUN_NAME = f"idd_{TRAIN_PRESET}_yolov8n"
RUN_DIR = PROJECT / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Проект:          {ROOT}")
print(f"Устройство:      {DEVICE}")
print(f"PyTorch:         {torch.__version__}")
print(f"Изображение:     {IMGSZ}px, batch={BATCH}, epochs={EPOCHS}")

## 1. Подготовка датасета

COCO-аннотации `_annotations.coco.json` конвертируются в YOLO-разметку
(`class cx cy w h` в нормализованных координатах). Изображения не копируются,
а подключаются символическими ссылками.

Для пресета `passport_fields` берутся только изображения паспортного разворота
(с рамкой `photo`) и делается новый split 80/10/10 — потому что в исходных
`valid`/`test` этих классов нет.

In [ ]:
if FORCE_REPREPARE or not DATA_YAML.exists():
    cmd = [
        sys.executable,
        str(PREPARE_SCRIPT),
        "--source", str(SOURCE_DATASET),
        "--output", str(DATASET_DIR),
        "--preset", TRAIN_PRESET,
        "--force",
    ]
    if TRAIN_PRESET != "all":
        cmd.append("--resplit")
    subprocess.run(cmd, check=True)
else:
    print(f"Датасет уже подготовлен: {DATA_YAML}")

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
NAMES = {int(k): v for k, v in cfg["names"].items()}
print(f"\nКлассов: {len(NAMES)}")

rows = []
for split in ("train", "val", "test"):
    images = list((DATASET_DIR / "images" / split).glob("*.*"))
    labels = list((DATASET_DIR / "labels" / split).glob("*.txt"))
    rows.append({"split": split, "images": len(images), "labels": len(labels)})
display(pd.DataFrame(rows))

print("\nКлассы:")
for i, name in NAMES.items():
    print(f"{i:2d}: {name}")

## 2. Распределение классов

Смотрим, насколько сбалансирован датасет. Редкие классы могут требовать
дообучения или объединения синонимов (например, `mrz`, `mrz_line0`, `mrz_line1`).

In [ ]:
counts = np.zeros(len(NAMES), dtype=int)
for split in ("train", "val", "test"):
    for label_file in (DATASET_DIR / "labels" / split).glob("*.txt"):
        for line in label_file.read_text(encoding="utf-8").splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1

dist = pd.DataFrame({"class": [NAMES[i] for i in range(len(NAMES))], "boxes": counts})
dist = dist.sort_values("boxes", ascending=True)

fig, ax = plt.subplots(figsize=(10, 13))
ax.barh(dist["class"], dist["boxes"], color="#2b7bba")
ax.set_xlabel("Количество аннотаций")
ax.set_title("Распределение классов (train + valid + test)")
for y, value in enumerate(dist["boxes"]):
    ax.text(value, y, f" {value}", va="center", fontsize=7)
plt.tight_layout()
plt.show()

display(dist.sort_values("boxes", ascending=False).reset_index(drop=True))

## 3. Визуализация ground truth

Случайные изображения train-выборки с нанесёнными рамками из YOLO-разметки.

In [ ]:
def class_color(index):
    hue = int(179 * index / max(1, len(NAMES)))
    hsv = np.uint8([[[hue, 220, 230]]])
    b, g, r = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0]
    return int(b), int(g), int(r)


COLORS = {i: class_color(i) for i in NAMES}


def draw_yolo_boxes(image_path, label_path, names, min_conf=None):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    label_path = Path(label_path)
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            color = COLORS.get(cls, (0, 255, 0))
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            label = names.get(cls, str(cls))
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
            cv2.rectangle(image, (x1, max(0, y1 - th - 6)), (x1 + tw + 4, y1), color, -1)
            cv2.putText(
                image, label, (x1 + 2, max(th + 2, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA,
            )
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


train_images = sorted((DATASET_DIR / "images" / "train").glob("*.jpg"))
samples = random.Random(SEED).sample(train_images, min(6, len(train_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, image_path in zip(axes.ravel(), samples):
    label_path = DATASET_DIR / "labels" / "train" / f"{image_path.stem}.txt"
    ax.imshow(draw_yolo_boxes(image_path, label_path, NAMES))
    ax.set_title(image_path.name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Обучение

Запускаем обучение YOLOv8. Чекпоинты, кривые и артефакты сохраняются в
`runs/detect/idd_yolov8n/`. Лучшие веса — `weights/best.pt`.

> В WSL2 для валидации Ultralytics создаёт `workers * 2` процессов. При большом числе
> workers это приводило к `CUDA error: out of memory` в `pin_memory`, поэтому в WSL
> автоматически выставляется `WORKERS = 0` и `BATCH = 8`.

In [ ]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,
    patience=20,
    plots=True,
    val=True,
    verbose=True,
)

print(f"\nАртефакты обучения: {RUN_DIR}")
print(f"Лучшие веса:        {BEST_WEIGHTS}")
print(f"Веса существуют:    {BEST_WEIGHTS.exists()}")

## 5. Кривые обучения

`results.png` содержит динамику loss и метрик, `results.csv` — числовые значения
по эпохам.

In [ ]:
def show_image(path, title=None, figsize=(14, 9)):
    path = Path(path)
    if not path.exists():
        print(f"Файл не найден: {path}")
        return
    image = Image.open(path)
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    plt.title(title or path.name)
    plt.show()


show_image(RUN_DIR / "results.png", "Кривые обучения YOLOv8")
show_image(RUN_DIR / "labels.jpg", "Примеры разметки после аугментаций")

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    display(history.tail(10))

## 6. Валидация

Считаем метрики на отложенной `valid`-выборке: Precision, Recall, mAP@50,
mAP@50-95 и метрики по каждому классу.

In [ ]:
def per_class_table(metrics, names):
    box = metrics.box
    index = np.asarray(getattr(box, "ap_class_index", np.arange(len(names))), dtype=int)

    def values(attr):
        value = np.asarray(getattr(box, attr))
        if value.size == 0:
            return np.full(len(index), np.nan)
        return value[index] if value.size > index.max() else value

    return (
        pd.DataFrame(
            {
                "class": [names[i] for i in index],
                "precision": values("p"),
                "recall": values("r"),
                "mAP50": values("ap50"),
                "mAP50-95": values("ap"),
            }
        )
        .sort_values("mAP50-95", ascending=False)
        .reset_index(drop=True)
    )


best = YOLO(str(BEST_WEIGHTS)) if BEST_WEIGHTS.exists() else model

val_metrics = best.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT),
    name=f"{RUN_NAME}_val",
    exist_ok=True,
    plots=True,
    verbose=False,
)

summary = pd.DataFrame(
    {
        "metric": ["Precision", "Recall", "mAP@50", "mAP@50-95"],
        "value": [
            val_metrics.box.mp,
            val_metrics.box.mr,
            val_metrics.box.map50,
            val_metrics.box.map,
        ],
    }
)
display(summary.style.format({"value": "{:.4f}"}))

print("Метрики по классам (valid):")
display(per_class_table(val_metrics, NAMES).style.format(
    {"precision": "{:.3f}", "recall": "{:.3f}", "mAP50": "{:.3f}", "mAP50-95": "{:.3f}"}
).background_gradient(subset=["mAP50-95"], cmap="Blues"))

val_dir = Path(val_metrics.save_dir)
show_image(val_dir / "confusion_matrix_normalized.png", "Матрица ошибок (normalized)")
show_image(val_dir / "BoxPR_curve.png", "Precision-Recall кривая")
show_image(val_dir / "BoxF1_curve.png", "F1-кривая")

## 7. Тестирование

Финальная оценка на `test`-выборке, которая не использовалась при обучении
и валидации.

In [ ]:
test_metrics = best.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
    plots=True,
    verbose=False,
)

test_summary = pd.DataFrame(
    {
        "metric": ["Precision", "Recall", "mAP@50", "mAP@50-95"],
        "value": [
            test_metrics.box.mp,
            test_metrics.box.mr,
            test_metrics.box.map50,
            test_metrics.box.map,
        ],
    }
)
display(test_summary.style.format({"value": "{:.4f}"}))

print("Метрики по классам (test):")
display(per_class_table(test_metrics, NAMES).style.format(
    {"precision": "{:.3f}", "recall": "{:.3f}", "mAP50": "{:.3f}", "mAP50-95": "{:.3f}"}
).background_gradient(subset=["mAP50-95"], cmap="Blues"))

show_image(Path(test_metrics.save_dir) / "confusion_matrix_normalized.png", "Матрица ошибок (test)")

## 8. Визуализация предсказаний

Инференс на случайных изображениях из `test`: рамки, классы и уверенность модели.

In [ ]:
test_images = sorted((DATASET_DIR / "images" / "test").glob("*.jpg"))
if not test_images:
    test_images = sorted((DATASET_DIR / "images" / "val").glob("*.jpg"))

predict_samples = random.Random(SEED).sample(test_images, min(6, len(test_images)))

predictions = best.predict(
    source=[str(p) for p in predict_samples],
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.45,
    device=DEVICE,
    save=True,
    project=str(PROJECT),
    name=f"{RUN_NAME}_predict",
    exist_ok=True,
    verbose=False,
)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
for ax, result in zip(axes.ravel(), predictions):
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(
        f"{Path(result.path).name} | найдено: {len(result.boxes)}", fontsize=9
    )
    ax.axis("off")
for ax in axes.ravel()[len(predictions):]:
    ax.axis("off")
plt.tight_layout()
out_path = PROJECT / f"{RUN_NAME}_predictions_grid.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Сохранено: {out_path}")

## 9. Быстрый инференс на одном изображении

Замените путь на своё изображение. Модель вернёт классы и координаты рамок,
которые можно использовать в OCR-пайплайне.

In [ ]:
image_path = predict_samples[0]
result = best.predict(source=str(image_path), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)[0]

rows = []
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    rows.append(
        {
            "class": NAMES[int(box.cls)],
            "confidence": float(box.conf),
            "x1": round(x1, 1),
            "y1": round(y1, 1),
            "x2": round(x2, 1),
            "y2": round(y2, 1),
        }
    )

display(pd.DataFrame(rows).sort_values("confidence", ascending=False))
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## 10. Экспорт модели (опционально)

ONNX удобен для запуска без PyTorch (например, через `onnxruntime`).

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, device=DEVICE)
print(f"ONNX-модель: {onnx_path}")